# Elastix Registration & Folding Correction

End-to-end example: register a pair of 2D images using
[ITKElastix](https://github.com/InsightSoftwareConsortium/ITKElastix)
(the ITK Python wrapper for Elastix), extract the displacement field,
and correct any negative Jacobian determinants with DVFopt.

Elastix supports multiple transform models.  We test:

1. **B-spline** — free-form deformation, commonly produces folding
2. **B-spline (aggressive)** — reduced regularisation to force more folding

**Requirements:**
```
pip install itk-elastix
```

In [ ]:
import time
import importlib

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D, iterative_parallel
from dvfopt.viz import (
    plot_grid_before_after,
    plot_checkerboard_before_after,
    plot_neg_jdet_neighborhoods,
)

HAS_ITK = importlib.util.find_spec("itk") is not None
print(f"itk-elastix: {'available' if HAS_ITK else 'MISSING — pip install itk-elastix'}")

## Configuration

In [ ]:
IMAGE_SIZE = 128
JDET_THRESHOLD = 0.01

# Real image paths (set both to use real data instead of synthetic)
FIXED_IMAGE_PATH = None
MOVING_IMAGE_PATH = None

## Generate (or load) test images

In [ ]:
def make_test_image(size, shapes):
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    for cy, cx, ry, rx, intensity in shapes:
        mask = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1.0
        img[mask] = intensity
    return img


if FIXED_IMAGE_PATH is not None and MOVING_IMAGE_PATH is not None:
    import SimpleITK as sitk
    fixed_np = sitk.GetArrayFromImage(sitk.ReadImage(FIXED_IMAGE_PATH, sitk.sitkFloat32))
    moving_np = sitk.GetArrayFromImage(sitk.ReadImage(MOVING_IMAGE_PATH, sitk.sitkFloat32))
    IMAGE_SIZE = fixed_np.shape[0]
    print(f"Loaded real images: {fixed_np.shape}")
else:
    S = IMAGE_SIZE
    fixed_np = make_test_image(S, [
        (S*0.35, S*0.35, S*0.15, S*0.12, 0.9),
        (S*0.60, S*0.55, S*0.18, S*0.14, 0.7),
        (S*0.30, S*0.65, S*0.10, S*0.10, 0.5),
        (S*0.70, S*0.30, S*0.12, S*0.08, 0.6),
    ])
    moving_np = make_test_image(S, [
        (S*0.30, S*0.40, S*0.18, S*0.10, 0.9),
        (S*0.65, S*0.50, S*0.15, S*0.17, 0.7),
        (S*0.25, S*0.60, S*0.12, S*0.08, 0.5),
        (S*0.72, S*0.25, S*0.10, S*0.12, 0.6),
    ])
    print(f"Generated synthetic {S}x{S} test images")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(fixed_np, cmap="gray"); axes[0].set_title("Fixed")
axes[1].imshow(moving_np, cmap="gray"); axes[1].set_title("Moving")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## Helpers

In [ ]:
def to_dvfopt(dy, dx):
    H, W = dy.shape
    deformation = np.zeros((3, 1, H, W), dtype=np.float64)
    deformation[1, 0] = dy
    deformation[2, 0] = dx
    return deformation


def summarize_and_correct(deformation, label, reg_time):
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg = int((jac_init <= 0).sum())
    H, W = deformation.shape[-2:]

    print(f"\n{'='*70}")
    print(f"  {label}  |  {H}x{W}  |  neg Jdet: {n_neg}  |  reg time: {reg_time:.2f}s")
    print(f"{'='*70}")

    if n_neg == 0:
        print("  No folding detected — skipping correction.")
        print(f"  min Jdet: {jac_init.min():+.6f}")
        return None

    t0 = time.perf_counter()
    phi = iterative_parallel(deformation.copy(), verbose=1, threshold=JDET_THRESHOLD)
    corr_time = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    print(f"  Correction: {corr_time:.2f}s  |  neg: {n_neg} -> {n_neg_final}  "
          f"|  min Jdet: {jac_init.min():+.4f} -> {jac_final.min():+.4f}  |  L2: {l2:.4f}")

    plot_grid_before_after(deformation, phi, title=label)
    plot_checkerboard_before_after(deformation, phi, title=label)
    plot_neg_jdet_neighborhoods(deformation, phi, title=label)
    return phi

---
## Elastix B-spline Registration

We use the ITK-Elastix Python API with a standard B-spline parameter map,
then extract the displacement field from the resulting transform.

In [ ]:
if HAS_ITK:
    import itk

    fixed_itk = itk.GetImageFromArray(fixed_np)
    moving_itk = itk.GetImageFromArray(moving_np)

    # --- Standard B-spline ---
    param_map = itk.ParameterObject.New()
    default_bspline = param_map.GetDefaultParameterMap("bspline")
    default_bspline["MaximumNumberOfIterations"] = ["500"]
    default_bspline["FinalGridSpacingInPhysicalUnits"] = ["16.0"]
    param_map.AddParameterMap(default_bspline)

    elastix_obj = itk.ElastixRegistrationMethod.New(
        fixed_itk, moving_itk
    )
    elastix_obj.SetParameterObject(param_map)
    elastix_obj.SetLogToConsole(False)

    t0 = time.perf_counter()
    elastix_obj.UpdateLargestPossibleRegion()
    reg_time_std = time.perf_counter() - t0

    result_transform = elastix_obj.GetTransformParameterObject()

    # Convert to displacement field via transformix
    transformix = itk.TransformixFilter.New(fixed_itk)
    transformix.SetTransformParameterObject(result_transform)
    transformix.SetComputeDeformationField(True)
    transformix.UpdateLargestPossibleRegion()
    disp_itk = transformix.GetOutputDeformationField()

    disp_np = itk.GetArrayFromImage(disp_itk)  # (H, W, 2)
    # ITK convention: [..., 0]=dx, [..., 1]=dy
    dx = disp_np[..., 0].astype(np.float64)
    dy = disp_np[..., 1].astype(np.float64)
    deformation_std = to_dvfopt(dy, dx)

    print(f"Standard B-spline registered in {reg_time_std:.2f}s")
    phi_std = summarize_and_correct(deformation_std, "Elastix B-spline", reg_time_std)
else:
    print("itk-elastix not installed — skipping")

## Elastix B-spline (aggressive — reduced regularisation)

Lower the grid spacing and reduce the regularisation weight to force
more folding.  This stress-tests DVFopt's ability to handle severe cases.

In [ ]:
if HAS_ITK:
    param_map_agg = itk.ParameterObject.New()
    agg_bspline = param_map_agg.GetDefaultParameterMap("bspline")
    agg_bspline["MaximumNumberOfIterations"] = ["500"]
    agg_bspline["FinalGridSpacingInPhysicalUnits"] = ["8.0"]  # finer grid
    agg_bspline["Metric0Weight"] = ["1.0"]
    agg_bspline["Metric1Weight"] = ["0.01"]  # reduce bending energy penalty
    param_map_agg.AddParameterMap(agg_bspline)

    elastix_agg = itk.ElastixRegistrationMethod.New(
        fixed_itk, moving_itk
    )
    elastix_agg.SetParameterObject(param_map_agg)
    elastix_agg.SetLogToConsole(False)

    t0 = time.perf_counter()
    elastix_agg.UpdateLargestPossibleRegion()
    reg_time_agg = time.perf_counter() - t0

    result_transform_agg = elastix_agg.GetTransformParameterObject()

    transformix_agg = itk.TransformixFilter.New(fixed_itk)
    transformix_agg.SetTransformParameterObject(result_transform_agg)
    transformix_agg.SetComputeDeformationField(True)
    transformix_agg.UpdateLargestPossibleRegion()
    disp_agg = itk.GetArrayFromImage(transformix_agg.GetOutputDeformationField())

    dx = disp_agg[..., 0].astype(np.float64)
    dy = disp_agg[..., 1].astype(np.float64)
    deformation_agg = to_dvfopt(dy, dx)

    print(f"Aggressive B-spline registered in {reg_time_agg:.2f}s")
    phi_agg = summarize_and_correct(deformation_agg, "Elastix B-spline (aggressive)", reg_time_agg)
else:
    print("itk-elastix not installed — skipping")

## Summary

In [ ]:
if HAS_ITK:
    for label, deformation, rt in [
        ("Standard B-spline", deformation_std, reg_time_std),
        ("Aggressive B-spline", deformation_agg, reg_time_agg),
    ]:
        phi = np.stack([deformation[1, 0], deformation[2, 0]])
        jac = jacobian_det2D(phi)
        n_neg = int((jac <= 0).sum())
        print(f"  {label:<30s}  reg={rt:.2f}s  neg_jdet={n_neg:>5d}  "
              f"min_jdet={jac.min():+.6f}")